In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Bokeh plotting library and functions. Note, this is probably an overkill but I have used most of these in my other projects.
from bokeh.io import output_notebook, show
from bokeh.models.annotations.labels import Label
from bokeh.layouts import column, row
from bokeh.models import ColumnDataSource, CustomJS, Slider, Whisker, BoxAnnotation, Arrow, OpenHead, Span
from bokeh.plotting import figure, show, output_file, save
from bokeh.models import Legend, LinearAxis, Range1d, ColumnDataSource, LabelSet, HoverTool, DatetimeTickFormatter

from ecmwf.datastores import Client
import os
import time
import logging

In [5]:
current_dir = str(os.getcwd())

files_dir = current_dir + '/data/'

file_name = files_dir + 'cams_solar_rad_weather.csv'

logging.basicConfig(level="INFO")

client = Client()
client.check_authentication()  # optional check

dataset = "cams-solar-radiation-timeseries"
# request = {
#     "sky_type": "observed_cloud",
#     "location": {"longitude": -5.68297, "latitude": 39.56428},
#     "altitude": ["-999."],
#     "date": ["2014-12-31/2018-12-31"],
#     "time_step": "1hour",
#     "time_reference": "universal_time",
#     "data_format": "csv"
# }

request = {
    "sky_type": "observed_cloud",
    "location": {"longitude": -6.26031, "latitude": 37.43123},
    "altitude": ["-999."],
    "date": ["2014-12-31/2018-12-31"],
    "time_step": "1minute",
    "time_reference": "universal_time",
    "data_format": "csv_expert"
}

remote_job = client.submit(dataset, request)

while not remote_job.results_ready:
    # Update the status information
    remote_job.update()
    
    # Show the current status
    print(f"Status: {remote_job.status}")
    
    # If the job is finished but had an error
    if remote_job.status == "failed":
        print("❌ The request failed.")
        break
        
    # Wait for 10 seconds before checking again
    print("Waiting 10 seconds before checking again...")
    time.sleep(10)

# Download the data if it's ready
if remote_job.results_ready:
    remote_job.download(target=file_name)
    print("✅ Download complete!")

INFO:ecmwf.datastores.processing:Request ID is 4fc61a84-a0cd-4ee4-9504-34072fe20ab4
INFO:ecmwf.datastores.processing:status has been updated to accepted


Status: accepted
Waiting 10 seconds before checking again...
Status: accepted
Waiting 10 seconds before checking again...
Status: accepted
Waiting 10 seconds before checking again...


INFO:ecmwf.datastores.processing:status has been updated to running


Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds before checking again...
Status: running
Waiting 10 seconds befor

INFO:ecmwf.datastores.processing:status has been updated to successful
INFO:multiurl.base:Downloading https://object-store.os-api.cci2.ecmwf.int:443/cci2-prod-cache-2/2026-02-12/9347ea7147d2ca74ed1d27cc8b27dcc2.csv


9347ea7147d2ca74ed1d27cc8b27dcc2.csv:   0%|          | 0.00/554M [00:00<?, ?B/s]

✅ Download complete!


In [6]:
file_name = files_dir + 'cams_solar_rad_weather_example.csv'

def read_csv_with_header_last_comment(filepath, encoding="utf-8", comment_char="#", sep=";"):
    
    filepath = Path(filepath)

    # ---- pass 1: find the last comment line that contains the header ----
    last_header_line = None
    with filepath.open("r", encoding=encoding, newline="") as f:
        for line in f:
            s = line.strip()
            if s.startswith(comment_char):
                # remove leading "#" (and any following space)
                payload = s.lstrip(comment_char).strip()

                # skip empty comment lines like "#"
                if payload:
                    last_header_line = payload
            else:
                # first non-comment line -> comments are finished
                break

    if last_header_line is None:
        raise ValueError("No header found in the leading comment block.")

    # Split into column names
    colnames = [c.strip() for c in last_header_line.split(sep)]
    # Drop any empty column names (e.g., if line ends with ';')
    colnames = [c for c in colnames if c != ""]

    # ---- pass 2: read the data, skipping comment lines, using extracted header ----
    df = pd.read_csv(filepath, sep=sep, encoding=encoding, comment=comment_char, header=None, names=colnames)

    return df

# Example usage:
cams_seville_pd = read_csv_with_header_last_comment(file_name)
print(cams_seville_pd.head())
print(cams_seville_pd.columns)

                            Observation period  TOA  Clear sky GHI  \
0  2014-12-31T00:00:00.0/2014-12-31T00:01:00.0  0.0            0.0   
1  2014-12-31T00:01:00.0/2014-12-31T00:02:00.0  0.0            0.0   
2  2014-12-31T00:02:00.0/2014-12-31T00:03:00.0  0.0            0.0   
3  2014-12-31T00:03:00.0/2014-12-31T00:04:00.0  0.0            0.0   
4  2014-12-31T00:04:00.0/2014-12-31T00:05:00.0  0.0            0.0   

   Clear sky BHI  Clear sky DHI  Clear sky BNI  GHI  BHI  DHI  BNI  ...  \
0            0.0            0.0            0.0  0.0  0.0  0.0  0.0  ...   
1            0.0            0.0            0.0  0.0  0.0  0.0  0.0  ...   
2            0.0            0.0            0.0  0.0  0.0  0.0  0.0  ...   
3            0.0            0.0            0.0  0.0  0.0  0.0  0.0  ...   
4            0.0            0.0            0.0  0.0  0.0  0.0  0.0  ...   

     fvol    fgeo  albedo  Cloud optical depth  Cloud coverage  Cloud type  \
0  0.0602  0.0292     NaN                  NaN    

In [7]:
# Site location of solar power station near Seville
SITE_LAT  = 37.43123
SITE_LON  = -6.26031
NEAREST_CITY = "Seville"

# Seville (city) coordinates from Wikipedia
CITY_LAT = 37.39000
CITY_LON = -5.99000

def haversine_km(lat1, lon1, lat2, lon2):
    """
    Vectorized haversine distance (km).
    lat/lon can be scalars or numpy arrays.
    """
    R = 6371.0088  # mean Earth radius in km
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# --- 1) Extract end time (after "/") and convert to datetime ---
end_time_str = cams_seville_pd["Observation period"].astype(str).str.split("/", n=1).str[1]

utc_dt = pd.to_datetime(end_time_str, errors="coerce").dt.floor("s")  # removes decimals

# Store utc times as strings in the format "YYYY-MM-DD HH:MM:SS":
# utc_time_str = utc_dt.dt.strftime("%Y-%m-%d %H:%M:%S")

# --- 2) Build the new dataframe with selected columns ---
cols_keep = ["Clear sky GHI", "GHI", "Reliability", "Snow probability", "Cloud optical depth", "Cloud coverage", "Cloud type"]

cams_extra_seville_pd = cams_seville_pd.loc[:, cols_keep].copy()

# Put utc_time first
cams_extra_seville_pd.insert(0, "utc_time", utc_dt)

# --- 3) Add constant metadata columns ---
cams_extra_seville_pd["loc_lat"] = SITE_LAT
cams_extra_seville_pd["loc_long"] = SITE_LON
cams_extra_seville_pd["city_name"] = NEAREST_CITY

# --- 4) Distance to Seville city centre (km) ---
dist_km = haversine_km(SITE_LAT, SITE_LON, CITY_LAT, CITY_LON)
cams_extra_seville_pd["distance"] = dist_km

cams_extra_seville_pd.head()

,utc_time,Clear sky GHI,GHI,Reliability,Snow probability,Cloud optical depth,Cloud coverage,Cloud type,loc_lat,loc_long,city_name,distance
0,2014-12-31 00:01:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
1,2014-12-31 00:02:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
2,2014-12-31 00:03:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
3,2014-12-31 00:04:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064
4,2014-12-31 00:05:00,0.0,0.0,1.0,-1,NaN,-1,-1,37.43123,-6.26031,Seville,24.31064


In [12]:
# Let's visualize power data by plotting power generation/demand for the various power sources along with
# some optional weather features.

# Set up plotting figure. Set x-axis to "datetime" so that the date time can be displayed appropriately.
def explore_plots(dataframe, x_axis_column, y_axis_columns, x_axis_label, y_axis_label, title, feature_columns=None,
                  features_ylabel=None, p=None, normalize=False, other_colors=False, color_plot='black', color_features='black',
                  labels=None, features_labels=None, symbols=None, symbols_features=None, replaced_columns=None):

    """
    Function to create interactive exploratory plots.
    
    Parameters
    ----------
    dataframe : Pandas dataframe object
        The Pandas dataframe holding the data you want to plot.
    x_axis_column : string
        The name of the column you want to plot along the x-axis.
    y_axis_columns : list
        The names of the columns for the primary data you want to plot along the y-axis.
    x_axis_label : string
        The label to use for the x axis.
    y_axis_label : string
        The label to use for the y axis for the primary data. Note, all column data plotted will have the same y-axis scale
        and label name.
    title : string
        The title for the plot.

    Optional
    --------
    feature_columns : list or string
        The names of the columns you want to plot as additional features along secondary y-axis
        Default: None
    features_ylabel : list or string
        The secondary y-axis feature label/s. If a list greater than one item, list is concatenated into a single string.
        Default: None    
    p : bokeh plotting object
        A bokeh plotting object to add additional plotting objects to.
        Default: None
    normalize : Boolean
        Whether to normalize the data. Caution, might not work well for some features or when including more than one feature.
        Default: False
    other_colors : Boolean
        Whether to use your own colors (True) or to use color names as defined in this function (False).
        Default: False
    color_plot : string or list
        A string or list of colors to use for plotting the primary data. If list, must be length of number y_axis_columns.
        Default: 'black'
    color_features : string or list
        A string or list of colors to use for plotting the features data. If list, must be length of number feature_columns.
        Default: 'black'
    labels : list or string
        The labels to assign all the primary data from the y_axis_columns, should be list of length y_axis_columns if more than one
        primary data is being plotted.
        Default: None
    features_labels : list or string
        The labels to assign all the features data from feature_columns, should be list of length feature_columns if more than one
        feature is being plotted.
        Default: None
    symbols : string or list
        The plotting markers for the primary data. If list, must be length of number of y_axis_columns.
        Default: None
    symbols_features : string or list
        The plotting markers for the features data. If list, must be length of number of feature_columns.
        Default: None
    replaced_columns : Boolean
        Whether to plot the replaced primary data produced through cleaning. The appropriate columns must exist in the dataframe
        if True and given as 'replaced_' and the y_axis_columns, e.g., 'replaced_energy_usage_Wh' for replacements for 
        the 'energy_usage_Wh'.
        Default: False

    Returns
    -------
    p : bokeh object
        The Bokeh plotting object.
    """
    
    colors={'violet':'#6E36BB','pink':'#D8BAFF','blue':'#2480D0','cyan':'#00E6E6','green':'#1DD14B',
            'yellow':'#FFD700','orange':'#FF6600','dorange':'#DAA520','red':'#DD082C','black':'#000000',
            'grey':'#D0D0D0','dgrey':'#666666'}

    if p is None:
        p = figure(title=title, height=900, width=1600, x_axis_type="datetime",
                   tools="reset, hover, zoom_in, zoom_out, box_zoom, wheel_zoom, pan, save")
    
        p.title.text_font_size = '20pt'
        p.yaxis.axis_label = y_axis_label
        p.xaxis.axis_label_text_font_size = "20pt"
        p.xaxis.major_label_text_font_size = "20pt"
        p.xaxis.axis_label_text_font = "times"
        p.xaxis.axis_label_text_color = "black"
        p.xaxis.major_tick_in = 10
        p.xaxis.major_tick_out = 0
        p.xaxis.minor_tick_in = 4
        p.xaxis.minor_tick_out = 0
        p.xaxis.major_tick_line_width = 2
        p.xaxis.axis_label = x_axis_label
        p.yaxis.axis_label_text_font_size = "20pt"
        p.yaxis.major_label_text_font_size = "20pt"
        p.yaxis.axis_label_text_font = "times"
        p.yaxis.axis_label_text_color = "black"
        p.yaxis.major_tick_in = 10
        p.yaxis.major_tick_out = 0
        p.yaxis.minor_tick_in = 4
        p.yaxis.minor_tick_out = 0
        p.yaxis.major_tick_line_width = 2
        # Rotate labels for better readability
        p.xaxis.major_label_orientation = 120
        # Reduce the number of x-axis ticks to avoid crowding
        p.xaxis.ticker.desired_num_ticks = 8

        # Format the x-axis datetime labels.
        p.xaxis.formatter = DatetimeTickFormatter(
            minutes="%d-%m-%y %H:%M",
            hours="%d-%m-%y %H:%M",
            days="%d-%m-%y %H:%M",
            months="%d-%m-%y %H:%M",
            years="%d-%m-%y %H:%M"
        )

    if other_colors:
        color_plot_values = color_plot
    else:
        if isinstance(color_plot, list): 
            color_plot_values = [colors[c] for c in color_plot]
        else:
            color_plot_values = colors[color_plot]

    # Create a loop to plot each column data as given by y_axis_columns. Note that these columns will have the same y-range along
    # the primary y-axis.
    color_index = 0
    marker_index = 0
    label_index = 0

    # print('labels: ', labels)
    # print('markers: ', symbols)
    # print('colors: ', color_plot_values)
    
    for i, column in enumerate(y_axis_columns):
        label = labels[label_index]

        # Create a loop to plot the column data for each city in Spain
        for y, city in enumerate(dataframe['city_name'].unique()):

            if normalize:
                max_primary_data = dataframe.loc[dataframe['city_name'] == city, column].max()
            else:
                max_primary_data = 1

            # Only show the first column by default, hide others
            visible = True if i == 0 | y == 0 else False
            
            # Scatter plot with symbols.
            if symbols:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data, size=10,
                          marker=symbols[marker_index], color=color_plot_values[color_index], alpha=0.5,
                          legend_label=label + ' ' + str(city), visible=visible)
                
            # Add a line to connect the symbols
            p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                   dataframe.loc[dataframe['city_name'] == city, column]/max_primary_data,
                   line_width=2, color=color_plot_values[color_index], alpha=0.5, legend_label=label + ' ' + str(city),
                   visible=visible)

            if replaced_columns:
                p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                          dataframe.loc[dataframe['city_name'] == city, f'replaced_{column}']/max_primary_data,
                          size=10, color="red", marker="diamond", alpha=0.5, legend_label="Replaced " + label + ' ' + str(city),
                          visible=visible)

            # print('marker_index: ', marker_index)
            # print('color_index: ', color_index)
            # print('label_index: ', label_index)
        
            marker_index = marker_index + 1
            color_index = color_index + 1
        label_index = label_index + 1
        

    # Create a loop to plot the feature data, if given. Note, secondary y axis must be the same for all features!
    if feature_columns:

        if other_colors:
            color_plot_values = color_features
        else:
            if isinstance(color_features, list): 
                color_plot_values = [colors[c] for c in color_features]
            else:
                color_plot_values = colors[color_features]

        if normalize:
            max_features_data = dataframe[feature_columns].max().max()
        else:
            max_features_data = 1

        # Specify the secondary y-range for plotting the features data. All features data will be plotted on the same secondary
        # y-axis range, so find minimum and maximum values for the first feature
        y_range = p.y_range
        p.extra_y_ranges['features'] = y_range
        label_index = 0
        color_index = 0
        marker_index = 0

        for i, feature_column in enumerate(feature_columns):

            feature_label = features_labels[label_index]

            # Create a loop to plot the features for each city.
            for y, city in enumerate(dataframe['city_name'].unique()):

                if normalize:
                    max_features_data = dataframe.loc[dataframe['city_name'] == city, feature_column].max()
                else:
                    max_features_data = 1

                # Only show the first column by default, hide others
                visible = True if i == 0 | y == 0 else False
                
                # Scatter plot with symbols. Only plot the energy usage as a function of time for a specific city.
                if symbols_features:
                    p.scatter(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                              dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data,
                              y_range_name="features", size=10, marker=symbols_features[marker_index],
                              color=color_plot_values[color_index], alpha=0.5, legend_label=feature_label + ' ' + str(city),
                              visible=visible)
                # Add a line to connect the symbols
                p.line(dataframe.loc[dataframe['city_name'] == city, x_axis_column],
                       dataframe.loc[dataframe['city_name'] == city, feature_column]/max_features_data, y_range_name="features",
                       line_width=2, color=color_plot_values[color_index], alpha=0.5,
                       legend_label=feature_label + ' ' + str(city), visible=visible)
            
                marker_index = marker_index + 1
                color_index = color_index + 1
            label_index = label_index + 1

        # Add the secondary y-axis     
        secondary_y_axis = LinearAxis(y_range_name="features", axis_label=" ".join(features_ylabel))
        p.add_layout(secondary_y_axis, 'right')

        secondary_y_axis.axis_label_text_font_size = "20pt"  # Adjust label font size
        secondary_y_axis.major_label_text_font_size = "20pt"  # Adjust tick label font size
        secondary_y_axis.axis_label_text_font = "times"
        secondary_y_axis.axis_label_text_color = "black"
        secondary_y_axis.axis_label_text_font_size = "16pt"
        secondary_y_axis.axis_label_text_font_style = "normal"
        secondary_y_axis.major_tick_in = 10
        secondary_y_axis.major_tick_out = 0
        secondary_y_axis.minor_tick_in = 4
        secondary_y_axis.minor_tick_out = 0

    # Allow user to hide/show plot features.
    p.add_layout(Legend(), 'right')
    p.legend.click_policy="hide"
    p.legend.background_fill_alpha = 0.3
    p.legend.border_line_alpha = 0.2

    return(p)

In [11]:
p = explore_plots(cams_extra_seville_pd, 'utc_time', ["Clear sky GHI", "GHI"],
                  r"$$\mathrm{UTC\ Date\ \&\ Time\ (DD-MM-YY\ \ HH:MM)}$$", r"$$\mathrm{Irradiation\ (Wh/m)}$$",
                  'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time',
                  feature_columns=["Cloud optical depth", "Cloud coverage"],
                  features_ylabel=[r"$$\mathrm{Cloud\ optical\ depth\ and\ coverage}$$"], p=None, normalize=False, other_colors=False,
                  color_plot=['red', 'dorange'],
                  color_features=['blue', 'black'],
                  labels=['Clear Sky GHI', 'GHI with Clouds'], features_labels=['Cloud Optical Depth', 'Cloud Coverage'],
                  symbols=['star', 'triangle'],
                  symbols_features=['circle', 'square'])

show(p)

# Save plots.
direct_out = current_dir + '/output/exploratory/'

# Create output directory if it doesn't already exist
if not os.path.exists(direct_out):
    os.makedirs(direct_out)

filename_out = direct_out + '/Ground_irradiation_vs_clouds.html'

title = 'Irradiation Clear Sky and with Clouds as well as cloud features vs UTC Time'

save(p, filename_out, title=title)

/var/folders/wv/ww_f6bg15tv52kq7ghvh0w880000gp/T/ipykernel_31787/3500979379.py:25: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  save(p, filename_out, title=title)


'/Users/u8010412/Library/CloudStorage/Dropbox/Data_Science/Projects/Kaggle_energy_data/archive/output/exploratory/Ground_irradiation_vs_clouds.html'